In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

26/02/25 13:37:07 WARN Utils: Your hostname, Rob resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/25 13:37:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/25 13:37:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [11]:
!curl -L -o fhvhv_tripdata_2021-01.parquet \
https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  294M  100  294M    0     0  50.4M      0  0:00:05  0:00:05 --:--:-- 55.9M


In [12]:
!ls -lh fhvhv_tripdata_2021-01.parquet

-rw-r--r-- 1 roban roban 295M Feb 25 13:50 fhvhv_tripdata_2021-01.parquet


In [13]:
!wc -l fhvhv_tripdata_2021-01.parquet

1006794 fhvhv_tripdata_2021-01.parquet


In [16]:
df = spark.read.parquet("fhvhv_tripdata_2021-01.parquet")

In [24]:
df.createOrReplaceTempView("fhv")

spark.sql("""
    SELECT pickup_datetime, trip_miles
    FROM fhv
    LIMIT 10
""").show()

+-------------------+----------+
|    pickup_datetime|trip_miles|
+-------------------+----------+
|2021-01-01 00:33:44|      5.26|
|2021-01-01 00:55:19|      3.65|
|2021-01-01 00:23:56|      3.51|
|2021-01-01 00:42:51|      0.74|
|2021-01-01 00:48:14|       9.2|
|2021-01-01 00:06:59|     9.725|
|2021-01-01 00:50:00|     2.469|
|2021-01-01 00:14:30|     13.53|
|2021-01-01 00:22:54|       1.6|
|2021-01-01 00:40:12|       3.2|
+-------------------+----------+



In [18]:
df.count()

11908468

In [25]:
df.select(
    "pickup_datetime",
    "dropoff_datetime",
    "trip_miles",
    "PULocationID",
    "DOLocationID"
).show(5, truncate=False)

+-------------------+-------------------+----------+------------+------------+
|pickup_datetime    |dropoff_datetime   |trip_miles|PULocationID|DOLocationID|
+-------------------+-------------------+----------+------------+------------+
|2021-01-01 00:33:44|2021-01-01 00:49:07|5.26      |230         |166         |
|2021-01-01 00:55:19|2021-01-01 01:18:21|3.65      |152         |167         |
|2021-01-01 00:23:56|2021-01-01 00:38:05|3.51      |233         |142         |
|2021-01-01 00:42:51|2021-01-01 00:45:50|0.74      |142         |143         |
|2021-01-01 00:48:14|2021-01-01 01:08:42|9.2       |143         |78          |
+-------------------+-------------------+----------+------------+------------+
only showing top 5 rows



In [27]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_f

In [33]:
import pandas as pd

In [35]:
df.limit(10).toPandas()

,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0003,B02682,B02682,2021-01-01 00:28:09,2021-01-01 00:31:42,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,5.260,...,1.98,2.75,NaN,0.00,14.99,N,N,,N,N
1,HV0003,B02682,B02682,2021-01-01 00:45:56,2021-01-01 00:55:19,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,3.650,...,1.63,0.00,NaN,0.00,17.06,N,N,,N,N
2,HV0003,B02764,B02764,2021-01-01 00:21:15,2021-01-01 00:22:41,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,3.510,...,1.25,2.75,NaN,0.94,12.98,N,N,,N,N
3,HV0003,B02764,B02764,2021-01-01 00:39:12,2021-01-01 00:42:37,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,0.740,...,0.70,2.75,NaN,0.00,7.41,N,N,,N,N
4,HV0003,B02764,B02764,2021-01-01 00:46:11,2021-01-01 00:47:17,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,9.200,...,2.41,2.75,NaN,0.00,22.44,N,N,,N,N
5,HV0005,B02510,NaN,2021-01-01 00:04:00,NaT,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,9.725,...,2.49,2.75,NaN,0.00,28.90,N,N,N,N,N
6,HV0005,B02510,NaN,2021-01-01 00:40:06,NaT,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,2.469,...,2.22,0.00,NaN,0.00,15.01,N,N,N,N,N
7,HV0003,B02764,B02764,2021-01-01 00:10:36,2021-01-01 00:12:28,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,13.530,...,3.08,0.00,NaN,0.00,34.20,N,N,,N,N
8,HV0003,B02875,B02875,2021-01-01 00:21:17,2021-01-01 00:22:25,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,1.600,...,0.61,0.00,NaN,0.00,6.26,N,N,,N,N
9,HV0003,B02875,B02875,2021-01-01 00:36:57,2021-01-01 00:38:09,2021-01-01 00:40:12,2021-01-01 00:53:31,255,232,3.200,...,1.03,2.75,NaN,2.82,10.99,N,N,,N,N


In [40]:
df_pandas = df.limit(100).toPandas()
df_pandas.head()

,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0003,B02682,B02682,2021-01-01 00:28:09,2021-01-01 00:31:42,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,5.26,...,1.98,2.75,NaN,0.00,14.99,N,N,,N,N
1,HV0003,B02682,B02682,2021-01-01 00:45:56,2021-01-01 00:55:19,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,3.65,...,1.63,0.00,NaN,0.00,17.06,N,N,,N,N
2,HV0003,B02764,B02764,2021-01-01 00:21:15,2021-01-01 00:22:41,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,3.51,...,1.25,2.75,NaN,0.94,12.98,N,N,,N,N
3,HV0003,B02764,B02764,2021-01-01 00:39:12,2021-01-01 00:42:37,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,0.74,...,0.70,2.75,NaN,0.00,7.41,N,N,,N,N
4,HV0003,B02764,B02764,2021-01-01 00:46:11,2021-01-01 00:47:17,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,9.20,...,2.41,2.75,NaN,0.00,22.44,N,N,,N,N


In [41]:
df_spark = spark.createDataFrame(df_pandas)
df_spark.show()

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

In [42]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_f

In [46]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter (df.hvfhs_license_num == 'HV0003')

DataFrame[pickup_datetime: timestamp_ntz, dropoff_datetime: timestamp_ntz, PULocationID: bigint, DOLocationID: bigint]

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 36984)
Traceback (most recent call last):
  File "/usr/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.12/socketserver.py", line 761, in __init__
    self.handle()
  File "/home/roban/spark-project/.venv/lib/python3.12/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/home/roban/spark-project/.venv/lib/python3.12/site-packages/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/home/roban/spark-project/.venv/lib/python3.

In [45]:
!head -n 10 head.csv